In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [4, 3]
import MDAnalysis as mda
import mdtraj as mdt
import pandas as pd
from scipy.optimize import minimize
from scipy import stats
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

from sklearn.decomposition import PCA
import barnaba as bb
import copy
import os
import pickle
from time import time
import datetime

import itertools
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

In [ ]:
colvar_rew=np.loadtxt("data/colvar_rew")

In [ ]:
def Gamma(l,metadb,observables,e,sigma=0.0):
    l = np.array(l)
    weight_all = metadb / 2.5 
    weight_all -= np.dot(observables,l) # lamba*s inside the exponential of the maxent formula
    shift_all = np.max(weight_all) # shift to be subtracted to avoid numerical overflow problems in the exponential
    weight_all = np.exp(weight_all - shift_all)

    weight_mt = metadb / 2.5 
    shift_mt = np.max(weight_mt) # shift to be subtracted to avoid numerical overflow problems in the exponential
    weight_mt = np.exp(weight_mt - shift_mt)

    f = np.log(np.sum(weight_all) / np.sum(weight_mt)) + shift_all - shift_mt + np.dot(l,e)
    der = e - np.dot(weight_all,observables) / np.sum(weight_all) # derivative with respect to l
    reg=sigma**2 # regularization term
    f+=0.5*np.sum(reg*l**2)
    der+=reg*l
    
    return(f,der)

def weights_noprop(l,metadb,observables,e):
    l = np.array(l)
    weight_all = metadb / 2.5
    obs_prop=observables - e

    weight_all -= np.dot(obs_prop,l) # lamba*s inside the exponential of the maxent formula
    shift_all = np.max(weight_all) # shift to be subtracted to avoid numerical overflow problems in the exponential
    weight_all = np.exp(weight_all - shift_all) 

    return weight_all/ np.sum(weight_all)

## Data Preparation

In [ ]:
path="data/JACS/"

In [ ]:
C1sH1s=pd.read_csv(path+"RDC_C1sH1.csv", header=None)
C2H2=pd.read_csv(path+"RDC_C2H2.csv", header=None)
C5H5=pd.read_csv(path+"RDC_C5H5.csv", header=None)
C6H6=pd.read_csv(path+"RDC_C6H6.csv", header=None)
C8H8=pd.read_csv(path+"RDC_C8H8.csv", header=None)
N1H1=pd.read_csv(path+"RDC_N1H1.csv", header=None)
N3H3=pd.read_csv(path+"RDC_N3H3.csv", header=None)

In [ ]:
labels_h1s=["21h1s","22h1s","23h1s","27h1s","29h1s","30h1s","31h1s","32h1s","33h1s","34h1s","35h1s","39h1s","40h1s"]
labels_c2=["20c2","22c2","27c2"]
labels_c5=["23c5","24c5","25c5","30c5","31c5","39c5","40c5","41c5","42c5","44c5"]
labels_c6=["23c6","24c6","25c6","29c6","31c6","41c6"]
labels_c8=["18c8","20c8","21c8","22c8","26c8","27c8","28c8","32c8","33c8","34c8","35c8","36c8","43c8"]
labels_n1=["18n1","21n1","28n1","36n1","43n1"]
labels_n3=["38n3","42n3"]

In [ ]:
C1sH1s=C1sH1s.loc[:,1:].T
C2H2=C2H2.loc[:,1:].T
C5H5=C5H5.loc[:,1:].T
C6H6=C6H6.loc[:,1:].T
C8H8=C8H8.loc[:,1:].T
N1H1=N1H1.loc[:,1:].T
N3H3=N3H3.loc[:,1:].T

In [ ]:
C1sH1s.columns=labels_h1s
C2H2.columns=labels_c2
C5H5.columns=labels_c5
C6H6.columns=labels_c6
C8H8.columns=labels_c8
N1H1.columns=labels_n1
N3H3.columns=labels_n3

In [ ]:
md_e0=pd.concat([C1sH1s,C2H2,C5H5,C6H6,C8H8,N1H1,N3H3],axis=1)

In [ ]:
path_data="data/JACS/RDC_data/E0-wt/"


In [ ]:
C1sH1s=pd.read_csv(path_data+"RDC_C1sH1s.csv")["rdc"]
C2H2=pd.read_csv(path_data+"RDC_C2H2.csv")["rdc"]
C5H5=pd.read_csv(path_data+"RDC_C5H5.csv")["rdc"]
C6H6=pd.read_csv(path_data+"RDC_C6H6.csv")["rdc"]
C8H8=pd.read_csv(path_data+"RDC_C8H8.csv")["rdc"]
N1H1=pd.read_csv(path_data+"RDC_N1H1.csv")["rdc"]
N3H3=pd.read_csv(path_data+"RDC_N3H3.csv")["rdc"]

In [ ]:
rdc_e0=np.array(pd.concat([C1sH1s,C2H2,C5H5,C6H6,C8H8,-N1H1,-N3H3]))

In [ ]:
weights=np.exp((colvar_rew[:,-1]-np.max(colvar_rew[:,-1]))/2.5) 
weights=weights/np.sum(weights) 

In [ ]:
calc_avg = np.array(np.sum(md_e0*weights[:,np.newaxis],axis=0))

In [ ]:
exp_avg = np.array(rdc_e0)

In [ ]:
reg = LinearRegression(fit_intercept=False).fit(calc_avg.reshape(-1,1),exp_avg.reshape(-1,1))

In [ ]:
r_value = reg.score(calc_avg.reshape(-1,1),exp_avg.reshape(-1,1))


In [ ]:
slope,intercept = reg.coef_[0],reg.intercept_

In [ ]:
md_e0_scaled=np.array(md_e0)*slope

In [ ]:
rmse_pre=np.sqrt(mean_squared_error(rdc_e0, calc_avg))
rmse_post=np.sqrt(mean_squared_error(rdc_e0, np.sum(md_e0_scaled*weights[:,np.newaxis],axis=0)))

In [ ]:
print(rmse_pre,rmse_post)

### NatCom

In [ ]:
C1sH1s=pd.read_csv("data/natcom/C1sH1s.dat", sep=' ',skiprows=1, header=None)
C2sH2s=pd.read_csv("data/natcom/C2sH2s.dat", sep=' ',skiprows=1, header=None)
C3sH3s=pd.read_csv("data/natcom/C3sH3s.dat", sep=' ',skiprows=1, header=None)
C4sH4s=pd.read_csv("data/natcom/C4sH4s.dat", sep=' ',skiprows=1, header=None)
C2H2=pd.read_csv("data/natcom/C2H2.dat", sep=' ',skiprows=1, header=None)
C5H5=pd.read_csv("data/natcom/C5H5.dat", sep=' ',skiprows=1, header=None)
C5C6=pd.read_csv("data/natcom/C5C6.dat", sep=' ',skiprows=1, header=None)
C6H6=pd.read_csv("data/natcom/C6H6.dat", sep=' ',skiprows=1, header=None)
C8H8=pd.read_csv("data/natcom/C8H8.dat", sep=' ',skiprows=1, header=None)

In [ ]:
labels=["1818C1sH1s","2020C1sH1s","2121C1sH1s","4040C1sH1s","4343C1sH1s","2626C1sH1s","2727C1sH1s","2828C1sH1s","3737C1sH1s","3838C1sH1s","2323C2sH2s","2424C2sH2s","2020C3sH3s","2626C3sH3s","2323C4sH4s","2424C4sH4s","2525C4sH4s","2020C2H2","2727C2H2","4141C5H5","2424C5H5","2525C5H5","3939C5H5","4242C5C6","2323C5C6","2525C5C6","3737C5C6","3939C5C6","2323C6H6","2020C8H8","2121C8H8","4343C8H8","2626C8H8","2727C8H8","2828C8H8"]

In [ ]:
C1sH1s.drop(columns=[0], inplace=True)
C2sH2s.drop(columns=[0], inplace=True)
C3sH3s.drop(columns=[0], inplace=True)
C4sH4s.drop(columns=[0], inplace=True)
C2H2.drop(columns=[0], inplace=True)
C5H5.drop(columns=[0], inplace=True)
C5C6.drop(columns=[0], inplace=True)
C6H6.drop(columns=[0], inplace=True)
C8H8.drop(columns=[0], inplace=True)

In [ ]:
MD_data_wt=pd.concat([C1sH1s,C2sH2s,C3sH3s,C4sH4s,C2H2,C5H5,C5C6,C6H6,C8H8], axis=1)
MD_data_wt.columns=labels

In [ ]:
RDC_data_C1sH1s=pd.read_csv('data/natcom/RDC_data/RDC_C1sH1s.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C2sH2s=pd.read_csv('data/natcom/RDC_data/RDC_C2sH2s.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C3sH3s=pd.read_csv('data/natcom/RDC_data/RDC_C3sH3s.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C4sH4s=pd.read_csv('data/natcom/RDC_data/RDC_C4sH4s.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C2H2=pd.read_csv('data/natcom/RDC_data/RDC_C2H2.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C5H5=pd.read_csv('data/natcom/RDC_data/RDC_C5H5.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C5C6=pd.read_csv('data/natcom/RDC_data/RDC_C5C6.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C6H6=pd.read_csv('data/natcom/RDC_data/RDC_C6H6.csv', sep=',',skiprows=1, header=None)[8]
RDC_data_C8H8=pd.read_csv('data/natcom/RDC_data/RDC_C8H8.csv', sep=',',skiprows=1, header=None)[8]


In [ ]:
RDC_data_wt=np.concatenate([RDC_data_C1sH1s,RDC_data_C2sH2s,RDC_data_C3sH3s,RDC_data_C4sH4s,RDC_data_C2H2,RDC_data_C5H5,RDC_data_C5C6,RDC_data_C6H6,RDC_data_C8H8])

In [ ]:
calc_avg = np.array(np.sum(MD_data_wt*weights[:,np.newaxis],axis=0))

In [ ]:
exp_avg = np.array(RDC_data_wt)

In [ ]:
reg = LinearRegression(fit_intercept=False).fit(calc_avg.reshape(-1,1),exp_avg.reshape(-1,1))
r_value = reg.score(calc_avg.reshape(-1,1),exp_avg.reshape(-1,1))
slope,intercept = reg.coef_[0],reg.intercept_


In [ ]:
MD_data_wt_scaled=np.array(MD_data_wt)*slope

In [ ]:
rmse_pre=np.sqrt(mean_squared_error(RDC_data_wt, calc_avg))
rmse_post=np.sqrt(mean_squared_error(RDC_data_wt, np.sum(MD_data_wt_scaled*weights[:,np.newaxis],axis=0)))

In [ ]:
md_e0_scaled_natcom=MD_data_wt_scaled
md_e0_scaled_jacs=md_e0_scaled
rdc_e0_natcom=RDC_data_wt
rdc_e0_jacs=rdc_e0

In [ ]:
md_data=np.concatenate([md_e0_scaled_jacs,md_e0_scaled_natcom],axis=1)
rdc_data=np.concatenate([rdc_e0_jacs,rdc_e0_natcom],axis=0)

In [ ]:
sigmas = np.array([3,4,5,6,6.5,7,8,9,10,15,20])
n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
rmse_cv = {}

for σ in sigmas:
    rmses = []
    for train_idx, test_idx in kf.split(rdc_data):
        md_train = md_data[:, train_idx]
        rdc_train = rdc_data[train_idx]
        md_test  = md_data[:, test_idx]
        rdc_test = rdc_data[test_idx]
        x0 = np.zeros(len(train_idx))
        res = minimize(Gamma, x0, method='BFGS', tol=1e-10, jac=True,
            args=(weights, md_train, rdc_train, σ))
        w_train = weights_noprop(res.x, weights, md_train, rdc_train)
        pred_test = np.average(md_test, axis=0, weights=w_train)
        rmse_fold = np.sqrt(mean_squared_error(rdc_test, pred_test))
        rmses.append(rmse_fold)

    rmse_cv[σ] = np.mean(rmses)

best_sigma = min(rmse_cv, key=rmse_cv.get)

res_full = minimize(
    Gamma, np.zeros(len(rdc_data)), method='BFGS', tol=1e-10, jac=True,
    args=(weights, md_data, rdc_data, best_sigma)
)
w_full = weights_noprop(res_full.x, weights, md_data, rdc_data)

rmse_full = np.sqrt(mean_squared_error(rdc_data,np.average(md_data, axis=0, weights=w_full)))
kish_full = 1.0 / np.sum(w_full**2)
